# Ilaria — Faza 1 (corpus → tokenizare → antrenare pe H100)

Rulează celulele în ordine. Celulele 1–4 merg pe o sesiune **fără GPU** (nu consumă unități); celulele 5–6 au nevoie de **H100**. Totul se scrie pe Drive în `MyDrive/ilaria`, deci o sesiune întreruptă se reia de unde a rămas (shard-urile complete sunt sărite).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
mkdir -p /content/drive/MyDrive/ilaria/corpus /content/drive/MyDrive/ilaria/brain-a /content/drive/MyDrive/ilaria/brain-b
if [ ! -d /content/nexus ]; then git clone -q https://github.com/office233/Nexuscortex.git /content/nexus; fi
cd /content/nexus && git pull -q && git log --oneline -1
if ! /usr/local/go/bin/go version 2>/dev/null | grep -q go1.26; then
  wget -q https://go.dev/dl/go1.26.8.linux-amd64.tar.gz -O /tmp/go.tgz
  rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
fi
/usr/local/go/bin/go version
pip -q install datasets 2>&1 | tail -1
cp /content/nexus/data/tokenizer-ro-en-32k.json /content/drive/MyDrive/ilaria/tokenizer.json
cd /content/nexus && /usr/local/go/bin/go build -o /content/corpus-tokenize ./cmd/corpus-tokenize && echo 'corpus-tokenize built'
nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'no GPU in this session (fine for corpus work)'

## 2. Corpus pe Drive (1–3 ore, reluabil)

FineWeb-2 românesc (3 M documente), FineWeb-Edu (2 M), Wikipedia RO (300 k bucăți) și EN (500 k). ≈ 4 miliarde de tokeni. Se poate rula de mai multe ori: continuă de la ultimul shard complet.

In [ ]:
%%bash
cd /content/nexus
python forge/prepare_corpus.py --out-dir /content/drive/MyDrive/ilaria/corpus \
  --sources wiki_ro,wiki_en,fineweb2_ro,fineweb_edu \
  --max wiki_ro=300000 --max wiki_en=500000 --max fineweb2_ro=3000000 --max fineweb_edu=2000000 2>&1 | grep -v Warning
ls /content/drive/MyDrive/ilaria/corpus | head -50; du -sh /content/drive/MyDrive/ilaria/corpus

## 3. Tokenizare (paralel) + stream unic

Sare peste shard-urile deja tokenizate (există `.bin`).

In [ ]:
%%bash
cd /content/nexus
for f in /content/drive/MyDrive/ilaria/corpus/*.jsonl; do
  p="${f%.jsonl}"
  [ -f "$p.bin" ] && continue
  echo "$p"
done | xargs -P 8 -I{} sh -c '/content/corpus-tokenize -tokenizer /content/drive/MyDrive/ilaria/tokenizer.json -in {}.jsonl -out {} 2>&1 | tail -1'
python forge/concat_streams.py --out /content/drive/MyDrive/ilaria/train_stream \
  --prefix ro=/content/drive/MyDrive/ilaria/corpus/fineweb2_ro --prefix ro=/content/drive/MyDrive/ilaria/corpus/wiki_ro \
  --prefix en=/content/drive/MyDrive/ilaria/corpus/fineweb_edu --prefix en=/content/drive/MyDrive/ilaria/corpus/wiki_en
cat /content/drive/MyDrive/ilaria/train_stream.json

## 4. (H100) Măsurăm viteza: 100 de pași cu rețeta A

Schimbă runtime-ul pe **H100** înainte de această celulă și rulează din nou celulele 1–2 (sesiune nouă). Citește `tok/s` din ultimele linii.

In [ ]:
%%bash
cd /content/nexus
python forge/train_ilaria.py --data /content/drive/MyDrive/ilaria/train_stream --out /content/speedtest \
  --embed-dim 768 --heads 12 --layers 12 --ffn-dim 2688 --ctx 1024 --max-seq-len 1024 --rope --swiglu --dropout 0 \
  --batch 64 --accum 4 --steps 100 --warmup 20 --lr 6e-4 --min-lr 6e-5 --wd 0.1 --precision bf16 --compile --eval-every 100 2>&1 | tail -15

## 5. (H100) Rețeta A — Ilaria-130M, ~3 G tokeni

`STEPS` = tokeni disponibili / 262 144, plafonat la 11 500. Checkpoint pe Drive la fiecare 500 de pași; dacă sesiunea cade, rulează din nou cu `--resume` (linia comentată).

In [ ]:
%%bash
cd /content/nexus
TOK=$(python -c "import json;print(json.load(open('/content/drive/MyDrive/ilaria/train_stream.json'))['tokens'])")
STEPS=$(( TOK / 262144 )); [ $STEPS -gt 11500 ] && STEPS=11500
echo "tokens=$TOK steps=$STEPS"
RESUME=""; [ -f /content/drive/MyDrive/ilaria/brain-a/checkpoint.pt ] && RESUME="--resume /content/drive/MyDrive/ilaria/brain-a/checkpoint.pt"
python forge/train_ilaria.py --data /content/drive/MyDrive/ilaria/train_stream --out /content/drive/MyDrive/ilaria/brain-a \
  --embed-dim 768 --heads 12 --layers 12 --ffn-dim 2688 --ctx 1024 --max-seq-len 1024 --rope --swiglu --dropout 0 \
  --batch 64 --accum 4 --steps $STEPS --warmup 500 --lr 6e-4 --min-lr 6e-5 --wd 0.1 --precision bf16 --compile --eval-every 500 $RESUME

## 6. Rezultatul

`/content/drive/MyDrive/ilaria/brain-a/transformer.nxtf` + `tokenizer.json` → copiate pe PC în `data/forge/brain-a/` și rulate cu `go run -tags gpu ./cmd/nxtf-run -data-dir ./data/forge/brain-a -gpu -prompt "Ștefan cel Mare a fost"`.